# LJ Dev Commerce - Phase 4

# IV. Warehouse ETL

This notebook performs the Extract, Transform, Validate, and PostgreSQL Load process for the Warehouse dataset in the LJ Dev Commerce project.

The Warehouse dataset follows the established LJ Dev Commerce Raw Data → PostgreSQL workflow. The reusable workflow is consistent across datasets, while profiling findings, business rules, transformations, relationships, and database integrity checks are determined specifically from the Warehouse dataset and project documentation.

**Workflow:**

Raw Data → Profiling → Independent Analyst Review → Transformation → Validation → Clean CSV → Database-Ready Mapping → PostgreSQL Load → Reconciliation → Database Integrity Validation


# =========================================================
# 01. Extract
# =========================================================

## 1.1 Imports and Project Setup

In [ ]:
import pandas as pd

In [2]:
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1.2 Load and Preserve Warehouse Raw Dataset

Load the raw Warehouse source file and preserve it as the original source dataset.

The raw dataset must not be modified during the ETL process. Any profiling, cleaning, transformation, or validation work will use separate DataFrame copies where appropriate.


In [3]:
# =========================================================
# 1.2 Load and Preserve Warehouse Raw Dataset
# =========================================================

# Define the raw Warehouse dataset path
zoho_inventory_warehouses_raw_path = Path(
    "../data/04_Zoho_Inventory/zoho_inventory_warehouses.csv"
)

# Load the raw source dataset
zoho_inventory_warehouses_raw = pd.read_csv(zoho_inventory_warehouses_raw_path)

# Preserve the original raw dataset
zoho_inventory_warehouses_raw_original = zoho_inventory_warehouses_raw.copy(deep=True)

print("Warehouse raw dataset loaded successfully.")
print("Rows:", zoho_inventory_warehouses_raw.shape[0])
print("Columns:", zoho_inventory_warehouses_raw.shape[1])

print("\nColumn names:")
print(zoho_inventory_warehouses_raw.columns.tolist())

print("\nRaw dataset preserved:",
      zoho_inventory_warehouses_raw.equals(zoho_inventory_warehouses_raw_original))

display(zoho_inventory_warehouses_raw)

Warehouse raw dataset loaded successfully.
Rows: 3
Columns: 12

Column names:
['WarehouseCode', 'WarehouseName', 'WarehouseType', 'Address', 'City', 'Country', 'ManagerName', 'ActiveFlag', 'CreatedOn', 'CreatedByUser', 'ModifiedOn', 'ModifiedByUser']

Raw dataset preserved: True


,WarehouseCode,WarehouseName,WarehouseType,Address,City,Country,ManagerName,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,WH001,Dubai Main Warehouse,General Storage,"Al Quoz, Dubai",Dubai,UAE,Ahmed Saleh,True,2026-07-01,admin,2026-07-15,admin
1,WH002,Abu Dhabi Warehouse,General Storage,"Mussafah, Abu Dhabi",Abu Dhabi,UAE,Fatima Noor,True,2026-07-02,admin,2026-07-15,admin
2,WH003,Sharjah Returns Hub,Returns,Sharjah Industrial Area,Sharjah,UAE,Rashid Khan,True,2026-07-03,admin,2026-07-14,admin


## 1.3 Initial Raw Data Inspection

Perform an initial inspection of the raw Warehouse dataset before profiling.

This step establishes the dataset structure, current pandas data types, missing-value counts, and a sample of the raw records. No transformations are performed at this stage.


In [4]:
# =========================================================
# 1.3 Initial Raw Data Inspection
# =========================================================

print("Dataset shape:")
print(zoho_inventory_warehouses_raw.shape)

print("\nColumn names:")
print(zoho_inventory_warehouses_raw.columns.tolist())

print("\nCurrent pandas data types:")
print(zoho_inventory_warehouses_raw.dtypes)

print("\nMissing values:")
print(zoho_inventory_warehouses_raw.isnull().sum())

print("\nRaw Warehouse data:")
display(zoho_inventory_warehouses_raw)

Dataset shape:
(3, 12)

Column names:
['WarehouseCode', 'WarehouseName', 'WarehouseType', 'Address', 'City', 'Country', 'ManagerName', 'ActiveFlag', 'CreatedOn', 'CreatedByUser', 'ModifiedOn', 'ModifiedByUser']

Current pandas data types:
WarehouseCode     object
WarehouseName     object
WarehouseType     object
Address           object
City              object
Country           object
ManagerName       object
ActiveFlag          bool
CreatedOn         object
CreatedByUser     object
ModifiedOn        object
ModifiedByUser    object
dtype: object

Missing values:
WarehouseCode     0
WarehouseName     0
WarehouseType     0
Address           0
City              0
Country           0
ManagerName       0
ActiveFlag        0
CreatedOn         0
CreatedByUser     0
ModifiedOn        0
ModifiedByUser    0
dtype: int64

Raw Warehouse data:


,WarehouseCode,WarehouseName,WarehouseType,Address,City,Country,ManagerName,ActiveFlag,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,WH001,Dubai Main Warehouse,General Storage,"Al Quoz, Dubai",Dubai,UAE,Ahmed Saleh,True,2026-07-01,admin,2026-07-15,admin
1,WH002,Abu Dhabi Warehouse,General Storage,"Mussafah, Abu Dhabi",Abu Dhabi,UAE,Fatima Noor,True,2026-07-02,admin,2026-07-15,admin
2,WH003,Sharjah Returns Hub,Returns,Sharjah Industrial Area,Sharjah,UAE,Rashid Khan,True,2026-07-03,admin,2026-07-14,admin


## 1.4 Data Profiling

Profile the untouched raw Warehouse dataset using the reusable `data_profiler_v1` tool.

The profiler is used to identify potential data-quality issues, semantic field types, missing values, duplicates, formatting inconsistencies, and other detectable anomalies before any cleaning or transformation is performed.

Profiling results are treated as findings for investigation. The profiler does not make business decisions or automatically determine the correct transformation.


In [5]:
# =========================================================
# 1.4 Data Profiling
# =========================================================

import sys
import importlib

# Identify the project root
project_root = Path.cwd().parent

# Add the project root to the Python path if needed
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import and reload the reusable data profiler
from profiler import data_profiler_v1 as profiler

importlib.reload(profiler)

print("Profiler loaded successfully:")
print(profiler.__file__)

print("\nProfiler configuration:")
print(profiler.DEFAULT_CONFIG)

Profiler loaded successfully:
c:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\profiler\data_profiler_v1.py

Profiler configuration:
{'date_detection_threshold': 0.8, 'numeric_detection_threshold': 0.8, 'email_detection_threshold': 0.8, 'phone_detection_threshold': 0.8, 'categorical_unique_ratio': 0.2, 'outlier_iqr_multiplier': 1.5, 'required_columns': [], 'unique_columns': [], 'non_negative_columns': []}


In [6]:
# Run the complete profiler against the untouched raw Warehouse dataset

profile_results = profiler.profile_dataset(zoho_inventory_warehouses_raw)

print("\nWarehouse profiling completed successfully.")
print("\nProfiler sections:")
print(list(profile_results.keys()))


Warehouse profiling completed successfully.

Profiler sections:
['field_types', 'detection_details', 'general', 'text', 'categorical', 'numeric', 'date', 'patterns', 'issues', 'configuration']


## 1.5 Review Profiling Results

Review the complete Warehouse profiling results before making any transformation decisions.

The automated profiler identifies potential data-quality issues, but the results must be independently reviewed in the context of the Warehouse dataset and LJ Dev Commerce business requirements.

This review examines:

* Automated profiling findings
* Issues requiring independent analyst review
* Potential business-rule decisions
* Issues that may require transformation

No data is modified during this stage.


In [7]:
# =========================================================
# 1.5 Review Profiling Results
# =========================================================

for section_name, section_result in profile_results.items():
    
    print("\n" + "=" * 70)
    print(section_name.upper())
    print("=" * 70)
    
    print(section_result)


FIELD_TYPES
WarehouseCode     identifier
WarehouseName           text
WarehouseType           text
Address                 text
City                    text
Country                 text
ManagerName             text
ActiveFlag           boolean
CreatedOn               date
CreatedByUser           text
ModifiedOn              date
ModifiedByUser          text
Name: DetectedFieldType, dtype: object

DETECTION_DETAILS
{'WarehouseCode': {'DetectedType': 'identifier', 'Confidence': 'Medium', 'Evidence': 'Column name suggests identifier semantics'}, 'WarehouseName': {'DetectedType': 'text', 'Confidence': 'Low', 'Evidence': 'No stronger semantic pattern detected'}, 'WarehouseType': {'DetectedType': 'text', 'Confidence': 'Low', 'Evidence': 'No stronger semantic pattern detected'}, 'Address': {'DetectedType': 'text', 'Confidence': 'Low', 'Evidence': 'No stronger semantic pattern detected'}, 'City': {'DetectedType': 'text', 'Confidence': 'Low', 'Evidence': 'No stronger semantic pattern detected'

## 1.6 Review Profiler Issues

Review the issues identified by the automated profiler.

The full profiling output provides a broad assessment of the raw Warehouse dataset. This step focuses specifically on the detected issues so they can be investigated before any transformation decisions are made.

Profiler findings are treated as indicators for review and do not automatically determine the correct transformation.


In [8]:
# =========================================================
# 1.6 Review Profiler Issues
# =========================================================

warehouse_issues = profile_results["issues"]

display(warehouse_issues)

,Column,IssueType,Severity,Count,Description
0,WarehouseName,Whitespace,Low,1,Leading or trailing whitespace detected.


## 1.7 Independent Analyst Review

Perform an independent analyst review of the raw Warehouse dataset beyond the automated profiler findings.

The automated profiler findings must be reviewed together with additional analyst checks covering data quality, identifiers, business values, date logic, and datatype preparation requirements.

The review confirms:

* `WarehouseCode` values are unique and contain no duplicates.
* No exact duplicate rows are present.
* No missing values were identified.
* `WarehouseType`, location values, boolean values, and audit fields show no obvious inconsistencies requiring transformation.
* `ModifiedOn` is not earlier than `CreatedOn`.
* `WarehouseName` contains one leading or trailing whitespace issue requiring transformation.
* `CreatedOn` contains valid ISO date values but is currently stored as an `object` datatype in the raw DataFrame.
* `ModifiedOn` contains valid ISO date values but is currently stored as an `object` datatype in the raw DataFrame.

The date values themselves are valid. The required date transformations are datatype preparation steps for the clean DataFrame and future database loading.

No transformations are performed during this review stage.

The findings from this review will be used to define the approved transformations in the next section.


In [9]:
# =========================================================
# 1.7 Independent Analyst Review
# =========================================================

# ---------------------------------------------------------
# 1. Identifier Review
# ---------------------------------------------------------

print("Duplicate WarehouseCode values:",
      zoho_inventory_warehouses_raw["WarehouseCode"].duplicated().sum())

print("\nWarehouseCode values:")
print(zoho_inventory_warehouses_raw["WarehouseCode"].tolist())


# ---------------------------------------------------------
# 2. Business Value Review
# ---------------------------------------------------------

for column in [
    "WarehouseType",
    "City",
    "Country",
    "ActiveFlag",
    "CreatedByUser",
    "ModifiedByUser"
]:
    print(f"\n{column}:")
    print(zoho_inventory_warehouses_raw[column].value_counts(dropna=False))


# ---------------------------------------------------------
# 3. Date Relationship Review
# ---------------------------------------------------------

created_dates = pd.to_datetime(
    zoho_inventory_warehouses_raw["CreatedOn"],
    errors="coerce"
)

modified_dates = pd.to_datetime(
    zoho_inventory_warehouses_raw["ModifiedOn"],
    errors="coerce"
)

print("\nModifiedOn earlier than CreatedOn:",
      (modified_dates < created_dates).sum())


# ---------------------------------------------------------
# 4. Data Type Preparation Review
# ---------------------------------------------------------

print("\nRaw DataFrame data types:")
print(zoho_inventory_warehouses_raw.dtypes)

print("\nDate fields requiring datatype preparation:")

for column in ["CreatedOn", "ModifiedOn"]:
    print(
        f"{column}: "
        f"raw dtype = {zoho_inventory_warehouses_raw[column].dtype}, "
        f"target clean dtype = datetime64[ns]"
    )

Duplicate WarehouseCode values: 0

WarehouseCode values:
['WH001', 'WH002', 'WH003']

WarehouseType:
WarehouseType
General Storage    2
Returns            1
Name: count, dtype: int64

City:
City
Dubai        1
Abu Dhabi    1
Sharjah      1
Name: count, dtype: int64

Country:
Country
UAE    3
Name: count, dtype: int64

ActiveFlag:
ActiveFlag
True    3
Name: count, dtype: int64

CreatedByUser:
CreatedByUser
admin    3
Name: count, dtype: int64

ModifiedByUser:
ModifiedByUser
admin    3
Name: count, dtype: int64

ModifiedOn earlier than CreatedOn: 0

Raw DataFrame data types:
WarehouseCode     object
WarehouseName     object
WarehouseType     object
Address           object
City              object
Country           object
ManagerName       object
ActiveFlag          bool
CreatedOn         object
CreatedByUser     object
ModifiedOn        object
ModifiedByUser    object
dtype: object

Date fields requiring datatype preparation:
CreatedOn: raw dtype = object, target clean dtype = datetime6

## 1.8 Transformation Decisions

Define the approved transformations based on the automated profiling results and independent analyst review.

Only transformations supported by identified data-quality or datatype preparation requirements will be applied.

### Approved Transformations

| Column          | Finding                                            | Transformation                         |
| --------------- | -------------------------------------------------- | -------------------------------------- |
| `WarehouseName` | One leading or trailing whitespace issue detected  | Remove leading and trailing whitespace |
| `CreatedOn`     | Valid ISO date values currently stored as `object` | Convert to `datetime64[ns]`            |
| `ModifiedOn`    | Valid ISO date values currently stored as `object` | Convert to `datetime64[ns]`            |

### Fields Requiring No Transformation

`WarehouseCode`, `WarehouseType`, `Address`, `City`, `Country`, `ManagerName`, `ActiveFlag`, `CreatedByUser`, and `ModifiedByUser` do not currently require transformation based on the profiling and independent analyst review.

The raw Warehouse dataset will remain unchanged. Transformations will be applied only to a separate working copy.


# =========================================================
# 02. Transform
# =========================================================

## 2.1 Create Clean DataFrame

This section creates a clean working copy of the raw Warehouse
dataset.

The original raw DataFrame remains unchanged.

Approved transformations will be applied to the Clean DataFrame
in the next step.

In [10]:
# =========================================================
# 2.1 Create Warehouse Working Copy
# =========================================================

zoho_inventory_warehouses_clean = zoho_inventory_warehouses_raw.copy()

print("Warehouse working copy created successfully.")
print("Raw dataset shape:", zoho_inventory_warehouses_raw.shape)
print("Working dataset shape:", zoho_inventory_warehouses_clean.shape)

Warehouse working copy created successfully.
Raw dataset shape: (3, 12)
Working dataset shape: (3, 12)


## 2.2 Apply Approved Transformations

Apply only the transformations approved during the profiling and independent analyst review.

The transformations are performed on `zoho_inventory_warehouses_clean`. The original `zoho_inventory_warehouses_raw` dataset remains unchanged.

Approved transformations:

* Remove leading and trailing whitespace from `WarehouseName`
* Convert `CreatedOn` to `datetime64[ns]`
* Convert `ModifiedOn` to `datetime64[ns]`


In [11]:
# =========================================================
# 2.2 Apply Approved Transformations
# =========================================================

# Remove leading and trailing whitespace
zoho_inventory_warehouses_clean["WarehouseName"] = (
    zoho_inventory_warehouses_clean["WarehouseName"].str.strip()
)

# Convert date columns to datetime
zoho_inventory_warehouses_clean["CreatedOn"] = pd.to_datetime(
    zoho_inventory_warehouses_clean["CreatedOn"],
    errors="coerce"
)

zoho_inventory_warehouses_clean["ModifiedOn"] = pd.to_datetime(
    zoho_inventory_warehouses_clean["ModifiedOn"],
    errors="coerce"
)

print("Approved Warehouse transformations applied successfully.")

Approved Warehouse transformations applied successfully.


# =========================================================
# 03. Validate
# =========================================================

## 3.1 Validate Applied Transformations

Validate the Warehouse transformations before creating the clean CSV.

The validation confirms:

* Leading and trailing whitespace has been removed from `WarehouseName`
* `CreatedOn` has been converted to `datetime64[ns]`
* `ModifiedOn` has been converted to `datetime64[ns]`
* No unexpected row changes occurred during transformation


In [12]:
# =========================================================
# 3.1 Validate Applied Transformations
# =========================================================

# Row count validation
print("Raw rows:", len(zoho_inventory_warehouses_raw))
print("Clean rows:", len(zoho_inventory_warehouses_clean))
print(
    "Row count matches:",
    len(zoho_inventory_warehouses_raw) == len(zoho_inventory_warehouses_clean)
)

# Whitespace validation
warehouse_name_whitespace = (
    zoho_inventory_warehouses_clean["WarehouseName"]
    != zoho_inventory_warehouses_clean["WarehouseName"].str.strip()
).sum()

print("\nWarehouseName values with leading/trailing whitespace:",
      warehouse_name_whitespace)

# Datatype validation
print("\nClean DataFrame data types:")
print(zoho_inventory_warehouses_clean.dtypes)

print("\nCreatedOn is datetime:",
      pd.api.types.is_datetime64_any_dtype(
          zoho_inventory_warehouses_clean["CreatedOn"]
      ))

print("ModifiedOn is datetime:",
      pd.api.types.is_datetime64_any_dtype(
          zoho_inventory_warehouses_clean["ModifiedOn"]
      ))

Raw rows: 3
Clean rows: 3
Row count matches: True

WarehouseName values with leading/trailing whitespace: 0

Clean DataFrame data types:
WarehouseCode             object
WarehouseName             object
WarehouseType             object
Address                   object
City                      object
Country                   object
ManagerName               object
ActiveFlag                  bool
CreatedOn         datetime64[ns]
CreatedByUser             object
ModifiedOn        datetime64[ns]
ModifiedByUser            object
dtype: object

CreatedOn is datetime: True
ModifiedOn is datetime: True


## 3.2 Validate Data Preservation

Validate that the approved transformations did not unintentionally alter unaffected Warehouse data.

The validation compares the raw and cleaned datasets while accounting for the three approved transformations:

* `WarehouseName` whitespace removal
* `CreatedOn` datatype conversion
* `ModifiedOn` datatype conversion

All other business and audit field values should remain unchanged.


In [13]:
# =========================================================
# 3.2 Validate Data Preservation
# =========================================================

# Columns that should remain completely unchanged
unchanged_columns = [
    "WarehouseCode",
    "WarehouseType",
    "Address",
    "City",
    "Country",
    "ManagerName",
    "ActiveFlag",
    "CreatedByUser",
    "ModifiedByUser"
]

all_columns_match = True

for column in unchanged_columns:
    
    matches = (
        zoho_inventory_warehouses_raw[column].astype(str).reset_index(drop=True)
        ==
        zoho_inventory_warehouses_clean[column].astype(str).reset_index(drop=True)
    ).all()
    
    print(f"{column}: {matches}")
    
    if not matches:
        all_columns_match = False


print("\nAll unaffected columns preserved:", all_columns_match)

WarehouseCode: True
WarehouseType: True
Address: True
City: True
Country: True
ManagerName: True
ActiveFlag: True
CreatedByUser: True
ModifiedByUser: True

All unaffected columns preserved: True


# =========================================================
# 04. Clean CSV
# =========================================================

## 4.1 Export Clean Warehouse Dataset

Export the validated `zoho_inventory_warehouses_clean` DataFrame as the clean Warehouse dataset.

The clean CSV represents the completed data-cleaning stage and preserves the validated Warehouse records before database-ready column mapping begins.


In [14]:
# =========================================================
# 4.1 Export Clean Warehouse Dataset
# =========================================================

from pathlib import Path

clean_warehouse_path = Path(
    r"C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce"
    r"\data\04_Zoho_Inventory\clean\zoho_inventory_warehouses_clean.csv"
)

# Create the clean folder if it does not exist
clean_warehouse_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Export clean dataset
zoho_inventory_warehouses_clean.to_csv(
    clean_warehouse_path,
    index=False
)

print("Clean Warehouse CSV exported successfully.")
print("Path:", clean_warehouse_path)
print("Rows exported:", len(zoho_inventory_warehouses_clean))
print("Columns exported:", len(zoho_inventory_warehouses_clean.columns))

Clean Warehouse CSV exported successfully.
Path: C:\JEP\DATA ANALYST PORTFOLIO\Lj_Dev_Commerce\data\04_Zoho_Inventory\clean\zoho_inventory_warehouses_clean.csv
Rows exported: 3
Columns exported: 12


## 4.2 Verify Exported Clean Warehouse CSV

Reload the exported clean CSV and verify that it matches the validated clean DataFrame. This confirms that the persisted clean file is the official handoff artifact for the next pipeline stage.


In [15]:
# =========================================================
# 4.2 Verify Exported Clean Warehouse CSV
# =========================================================

zoho_inventory_warehouses_clean_csv = pd.read_csv(
    clean_warehouse_path,
    parse_dates=["CreatedOn", "ModifiedOn"]
)

print("Clean CSV rows:", len(zoho_inventory_warehouses_clean_csv))
print("Clean DataFrame rows:", len(zoho_inventory_warehouses_clean))
print(
    "Row count matches:",
    len(zoho_inventory_warehouses_clean_csv)
    == len(zoho_inventory_warehouses_clean)
)

print(
    "Column structure matches:",
    list(zoho_inventory_warehouses_clean_csv.columns)
    == list(zoho_inventory_warehouses_clean.columns)
)

print(
    "Values match validated clean DataFrame:",
    zoho_inventory_warehouses_clean_csv.equals(
        zoho_inventory_warehouses_clean
    )
)

print("\nClean CSV data types:")
print(zoho_inventory_warehouses_clean_csv.dtypes)


Clean CSV rows: 3
Clean DataFrame rows: 3
Row count matches: True
Column structure matches: True
Values match validated clean DataFrame: True

Clean CSV data types:
WarehouseCode             object
WarehouseName             object
WarehouseType             object
Address                   object
City                      object
Country                   object
ManagerName               object
ActiveFlag                  bool
CreatedOn         datetime64[ns]
CreatedByUser             object
ModifiedOn        datetime64[ns]
ModifiedByUser            object
dtype: object


# =========================================================
# 5. Database Ready
# =========================================================

## 5.1 Define Source → Target Mapping

The verified Clean Warehouse CSV uses Zoho Inventory source
column names.

This section defines the approved Source → Target Mapping from
the verified Clean Warehouse CSV to the PostgreSQL Warehouse
target structure.

The target column names are based on the LJ Dev Commerce
Enterprise Database Design Data Dictionary.

Mapping:

- WarehouseCode → warehouse_id
- WarehouseName → warehouse_name
- WarehouseType → warehouse_type
- Address → address
- City → city
- Country → country
- ManagerName → warehouse_manager
- ActiveFlag → is_active
- CreatedOn → created_date
- CreatedByUser → created_by
- ModifiedOn → updated_date
- ModifiedByUser → updated_by

Important:

- The exported and verified Clean CSV is the source for the
  Database-Ready process.
- The original raw dataset is not modified.
- WarehouseCode maps to warehouse_id based on the Warehouse
  Data Dictionary, where warehouse_id uses values such as WH001.

In [18]:
# =========================================================
# 5.1 Define Source → Target Mapping
# =========================================================

warehouse_source_to_target_mapping = {
    "WarehouseCode": "warehouse_id",
    "WarehouseName": "warehouse_name",
    "WarehouseType": "warehouse_type",
    "Address": "address",
    "City": "city",
    "Country": "country",
    "ManagerName": "warehouse_manager",
    "ActiveFlag": "is_active",
    "CreatedOn": "created_date",
    "CreatedByUser": "created_by",
    "ModifiedOn": "updated_date",
    "ModifiedByUser": "updated_by"
}

print("Warehouse Source → Target Mapping")
print("=" * 60)

for source_column, target_column in warehouse_source_to_target_mapping.items():
    print(f"{source_column:<20} → {target_column}")

Warehouse Source → Target Mapping
WarehouseCode        → warehouse_id
WarehouseName        → warehouse_name
WarehouseType        → warehouse_type
Address              → address
City                 → city
Country              → country
ManagerName          → warehouse_manager
ActiveFlag           → is_active
CreatedOn            → created_date
CreatedByUser        → created_by
ModifiedOn           → updated_date
ModifiedByUser       → updated_by


## 5.2 Create Database-Ready Dataset FROM EXPORTED CSV

This section creates the Database-Ready Warehouse dataset from
the exported and verified Clean Warehouse CSV.

The Clean CSV is used as the official source for the database
preparation stage.

Process:

Verified Clean CSV
        ↓
Apply Source → Target Mapping
        ↓
Create Database-Ready DataFrame
        ↓
zoho_inventory_warehouses_db_ready

The original raw dataset and Clean DataFrame are not modified.

In [19]:
# =========================================================
# 5.2 Create Database-Ready Dataset FROM EXPORTED CSV
# =========================================================

# Create Database-Ready Warehouse dataset
# from the exported and verified Clean CSV

zoho_inventory_warehouses_db_ready = (
    zoho_inventory_warehouses_clean_csv
    .rename(columns=warehouse_source_to_target_mapping)
    .copy()
)

print("Database-Ready Warehouse dataset created successfully.")

print("\nDataFrame:")
print("zoho_inventory_warehouses_db_ready")

print("\nShape:", zoho_inventory_warehouses_db_ready.shape)

print("\nColumns:")
print(zoho_inventory_warehouses_db_ready.columns.tolist())

print("\nData types:")
print(zoho_inventory_warehouses_db_ready.dtypes)

print("\nPreview:")
display(zoho_inventory_warehouses_db_ready.head())

Database-Ready Warehouse dataset created successfully.

DataFrame:
zoho_inventory_warehouses_db_ready

Shape: (3, 12)

Columns:
['warehouse_id', 'warehouse_name', 'warehouse_type', 'address', 'city', 'country', 'warehouse_manager', 'is_active', 'created_date', 'created_by', 'updated_date', 'updated_by']

Data types:
warehouse_id                 object
warehouse_name               object
warehouse_type               object
address                      object
city                         object
country                      object
warehouse_manager            object
is_active                      bool
created_date         datetime64[ns]
created_by                   object
updated_date         datetime64[ns]
updated_by                   object
dtype: object

Preview:


,warehouse_id,warehouse_name,warehouse_type,address,city,country,warehouse_manager,is_active,created_date,created_by,updated_date,updated_by
0,WH001,Dubai Main Warehouse,General Storage,"Al Quoz, Dubai",Dubai,UAE,Ahmed Saleh,True,2026-07-01,admin,2026-07-15,admin
1,WH002,Abu Dhabi Warehouse,General Storage,"Mussafah, Abu Dhabi",Abu Dhabi,UAE,Fatima Noor,True,2026-07-02,admin,2026-07-15,admin
2,WH003,Sharjah Returns Hub,Returns,Sharjah Industrial Area,Sharjah,UAE,Rashid Khan,True,2026-07-03,admin,2026-07-14,admin


## 5.3 Database-Ready Validation

This section validates the Database-Ready Warehouse dataset before
it is loaded into PostgreSQL.

Validation includes:

- Row count preservation
- Expected target columns and column order
- Required data types
- Missing values
- Primary key uniqueness and null validation
- Source → Target Mapping completeness

The Database-Ready dataset must successfully pass all required
validations before proceeding to PostgreSQL loading.

In [20]:
# =========================================================
# 5.3 Database-Ready Validation
# =========================================================

# ---------------------------------------------------------
# Expected PostgreSQL target columns
# ---------------------------------------------------------

expected_columns = [
    "warehouse_id",
    "warehouse_name",
    "warehouse_type",
    "address",
    "city",
    "country",
    "warehouse_manager",
    "is_active",
    "created_date",
    "created_by",
    "updated_date",
    "updated_by"
]


# ---------------------------------------------------------
# 1. Row Count Validation
# ---------------------------------------------------------

row_count_matches = (
    len(zoho_inventory_warehouses_db_ready)
    == len(zoho_inventory_warehouses_clean_csv)
)

print("1. Row Count Validation")
print(
    "Clean CSV rows:",
    len(zoho_inventory_warehouses_clean_csv)
)
print(
    "Database-Ready rows:",
    len(zoho_inventory_warehouses_db_ready)
)
print("Row count matches:", row_count_matches)


# ---------------------------------------------------------
# 2. Column Validation
# ---------------------------------------------------------

columns_match = (
    zoho_inventory_warehouses_db_ready.columns.tolist()
    == expected_columns
)

print("\n2. Column Validation")
print("Expected columns:")
print(expected_columns)

print("\nActual columns:")
print(
    zoho_inventory_warehouses_db_ready.columns.tolist()
)

print("\nColumns and order match:", columns_match)


# ---------------------------------------------------------
# 3. Required Datatype Validation
# ---------------------------------------------------------

print("\n3. Required Datatype Validation")

print(
    "is_active is bool:",
    str(
        zoho_inventory_warehouses_db_ready["is_active"].dtype
    ) == "bool"
)

print(
    "created_date is datetime:",
    pd.api.types.is_datetime64_any_dtype(
        zoho_inventory_warehouses_db_ready["created_date"]
    )
)

print(
    "updated_date is datetime:",
    pd.api.types.is_datetime64_any_dtype(
        zoho_inventory_warehouses_db_ready["updated_date"]
    )
)


# ---------------------------------------------------------
# 4. Missing Value Validation
# ---------------------------------------------------------

missing_values = (
    zoho_inventory_warehouses_db_ready
    .isnull()
    .sum()
)

print("\n4. Missing Value Validation")
print(missing_values)


# ---------------------------------------------------------
# 5. Primary Key Validation
# ---------------------------------------------------------

warehouse_id_is_unique = (
    zoho_inventory_warehouses_db_ready["warehouse_id"]
    .is_unique
)

warehouse_id_has_no_nulls = (
    zoho_inventory_warehouses_db_ready["warehouse_id"]
    .notna()
    .all()
)

print("\n5. Primary Key Validation")
print("warehouse_id is unique:", warehouse_id_is_unique)
print(
    "warehouse_id has no null values:",
    warehouse_id_has_no_nulls
)


# ---------------------------------------------------------
# 6. Source → Target Mapping Validation
# ---------------------------------------------------------

mapping_target_columns = list(
    warehouse_source_to_target_mapping.values()
)

mapping_is_complete = (
    mapping_target_columns == expected_columns
)

print("\n6. Source → Target Mapping Validation")
print("Mapping targets match expected columns:",
      mapping_is_complete)


# ---------------------------------------------------------
# Final Validation Summary
# ---------------------------------------------------------

all_validations_passed = all([
    row_count_matches,
    columns_match,
    warehouse_id_is_unique,
    warehouse_id_has_no_nulls,
    mapping_is_complete,
    str(
        zoho_inventory_warehouses_db_ready["is_active"].dtype
    ) == "bool",
    pd.api.types.is_datetime64_any_dtype(
        zoho_inventory_warehouses_db_ready["created_date"]
    ),
    pd.api.types.is_datetime64_any_dtype(
        zoho_inventory_warehouses_db_ready["updated_date"]
    )
])

print("\n" + "=" * 60)
print("DATABASE-READY VALIDATION SUMMARY")
print("=" * 60)

print(
    "All required validations passed:",
    all_validations_passed
)

1. Row Count Validation
Clean CSV rows: 3
Database-Ready rows: 3
Row count matches: True

2. Column Validation
Expected columns:
['warehouse_id', 'warehouse_name', 'warehouse_type', 'address', 'city', 'country', 'warehouse_manager', 'is_active', 'created_date', 'created_by', 'updated_date', 'updated_by']

Actual columns:
['warehouse_id', 'warehouse_name', 'warehouse_type', 'address', 'city', 'country', 'warehouse_manager', 'is_active', 'created_date', 'created_by', 'updated_date', 'updated_by']

Columns and order match: True

3. Required Datatype Validation
is_active is bool: True
created_date is datetime: True
updated_date is datetime: True

4. Missing Value Validation
warehouse_id         0
warehouse_name       0
warehouse_type       0
address              0
city                 0
country              0
warehouse_manager    0
is_active            0
created_date         0
created_by           0
updated_date         0
updated_by           0
dtype: int64

5. Primary Key Validation
wareh

# =========================================================
# 6. PostgreSQL
# =========================================================

## 6.1 Connect to PostgreSQL

This section establishes a connection to the LJ Dev Commerce
PostgreSQL database.

The connection configuration follows the same successful approach
used in the completed Supplier and Category ETL notebooks.

The Warehouse Database-Ready dataset has already been validated
and is now ready for PostgreSQL loading.

In [21]:
# =========================================================
# 6.1 Connect to PostgreSQL
# =========================================================

import psycopg2
from getpass import getpass


# ---------------------------------------------------------
# PostgreSQL Connection Configuration
# ---------------------------------------------------------

DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "lj_dev_commerce"
DB_USER = "postgres"

DB_PASSWORD = getpass("Enter PostgreSQL password: ")


# ---------------------------------------------------------
# Connect to PostgreSQL
# ---------------------------------------------------------

try:
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conn.cursor()

    print("PostgreSQL connection successful.")
    print("Database:", DB_NAME)
    print("Host:", DB_HOST)
    print("Port:", DB_PORT)

except Exception as e:
    print("PostgreSQL connection failed.")
    print("Error:", e)

PostgreSQL connection successful.
Database: lj_dev_commerce
Host: localhost
Port: 5432


## 6.2 Check / Create Warehouse Table

This section prepares the PostgreSQL Warehouse target table.

Before creating the table, we first check whether
`commerce.warehouse` already exists.

This prevents accidentally recreating an existing table.

If the table does not exist, it is created based on the approved
LJ Dev Commerce Warehouse Data Dictionary.

Target table:

`commerce.warehouse`

In [ ]:
# =========================================================
# 6.2 Check Whether Warehouse Table Exists
# =========================================================

cursor.execute("""
    SELECT EXISTS (
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = 'commerce'
          AND table_name = 'warehouse'
    );
""")

warehouse_table_exists = cursor.fetchone()[0]

print("Schema: commerce")
print("Table: warehouse")
print("Table exists:", warehouse_table_exists)

Schema: commerce
Table: warehouse
Table exists: True


## 6.2.1 Create Warehouse Table

The `commerce.warehouse` table does not currently exist.

This section creates the Warehouse PostgreSQL target table based
on the approved LJ Dev Commerce Warehouse Data Dictionary.

Key design decisions:

- `warehouse_id` is the Primary Key.
- All Data Dictionary fields are required and therefore use
  `NOT NULL`.
- `is_active` uses the PostgreSQL `BOOLEAN` datatype.
- `created_date` and `updated_date` use the PostgreSQL
  `TIMESTAMP` datatype.
- Warehouse is a parent/master table and does not contain
  foreign keys at this stage.

In [ ]:
# =========================================================
# 6.2.1 Create Warehouse Table
# =========================================================

create_warehouse_table_query = """
CREATE TABLE commerce.warehouse (

    warehouse_id TEXT PRIMARY KEY,

    warehouse_name TEXT NOT NULL,
    warehouse_type TEXT NOT NULL,

    address TEXT NOT NULL,
    city TEXT NOT NULL,
    country TEXT NOT NULL,

    warehouse_manager TEXT NOT NULL,

    is_active BOOLEAN NOT NULL,

    created_date TIMESTAMP NOT NULL,
    created_by TEXT NOT NULL,

    updated_date TIMESTAMP NOT NULL,
    updated_by TEXT NOT NULL

);
"""

try:
    cursor.execute(create_warehouse_table_query)
    conn.commit()

    print("Warehouse table created successfully.")
    print("Table: commerce.warehouse")

except Exception as e:
    conn.rollback()

    print("Warehouse table creation failed.")
    print("Error:", e)

Warehouse table created successfully.
Table: commerce.warehouse


## 6.3 Verify Structure

This section verifies the actual structure of the
`commerce.warehouse` table in PostgreSQL.

The verification checks:

- Column names
- Column order
- PostgreSQL data types
- Nullability

This confirms that the created PostgreSQL table matches the
approved Warehouse target structure before proceeding to primary
key verification and data loading.

In [25]:
# =========================================================
# 6.3 Verify Structure
# =========================================================

# Retrieve the actual Warehouse table structure

verify_structure_query = """
SELECT
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'commerce'
  AND table_name = 'warehouse'
ORDER BY ordinal_position;
"""

cursor.execute(verify_structure_query)

warehouse_table_structure = cursor.fetchall()


# Display the actual table structure

print("Warehouse Table Structure")
print("=" * 80)

for row in warehouse_table_structure:
    print(
        f"Position: {row[0]:<2} | "
        f"Column: {row[1]:<20} | "
        f"Type: {row[2]:<25} | "
        f"Nullable: {row[3]}"
    )

Warehouse Table Structure
Position: 1  | Column: warehouse_id         | Type: text                      | Nullable: NO
Position: 2  | Column: warehouse_name       | Type: text                      | Nullable: NO
Position: 3  | Column: warehouse_type       | Type: text                      | Nullable: NO
Position: 4  | Column: address              | Type: text                      | Nullable: NO
Position: 5  | Column: city                 | Type: text                      | Nullable: NO
Position: 6  | Column: country              | Type: text                      | Nullable: NO
Position: 7  | Column: warehouse_manager    | Type: text                      | Nullable: NO
Position: 8  | Column: is_active            | Type: boolean                   | Nullable: NO
Position: 9  | Column: created_date         | Type: timestamp without time zone | Nullable: NO
Position: 10 | Column: created_by           | Type: text                      | Nullable: NO
Position: 11 | Column: updated_date       

## 6.4 Verify Primary Key

This section verifies that `warehouse_id` is correctly configured
as the Primary Key of the `commerce.warehouse` table.

The verification confirms the primary key constraint and the
column assigned to it before records are inserted.

In [26]:
# =========================================================
# 6.4 Verify Primary Key
# =========================================================

# Retrieve the Primary Key constraint information

verify_primary_key_query = """
SELECT
    tc.constraint_name,
    kcu.column_name
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
WHERE tc.constraint_type = 'PRIMARY KEY'
    AND tc.table_schema = 'commerce'
    AND tc.table_name = 'warehouse';
"""

cursor.execute(verify_primary_key_query)

warehouse_primary_key = cursor.fetchall()


# Display the Primary Key information

print("Warehouse Primary Key")
print("=" * 60)

if warehouse_primary_key:
    for constraint_name, column_name in warehouse_primary_key:
        print("Constraint:", constraint_name)
        print("Primary Key Column:", column_name)
else:
    print("No Primary Key found.")

Warehouse Primary Key
Constraint: warehouse_pkey
Primary Key Column: warehouse_id


## 6.5 Insert Records

This section inserts the validated Database-Ready Warehouse dataset
into the `commerce.warehouse` PostgreSQL table.

The records are inserted from:

`zoho_inventory_warehouses_db_ready`

The Database-Ready dataset was created from the exported and
verified Clean Warehouse CSV.

The insert operation uses the approved Source → Target Mapping and
the verified PostgreSQL Warehouse table structure.

In [27]:
# =========================================================
# 6.5 Insert Records
# =========================================================

# Define the INSERT query

insert_warehouse_query = """
INSERT INTO commerce.warehouse (
    warehouse_id,
    warehouse_name,
    warehouse_type,
    address,
    city,
    country,
    warehouse_manager,
    is_active,
    created_date,
    created_by,
    updated_date,
    updated_by
)
VALUES (
    %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s
);
"""


# Convert the Database-Ready DataFrame into records

warehouse_records = list(
    zoho_inventory_warehouses_db_ready.itertuples(
        index=False,
        name=None
    )
)

print("Records prepared for insertion:", len(warehouse_records))


# Insert records into PostgreSQL

try:
    cursor.executemany(
        insert_warehouse_query,
        warehouse_records
    )

    conn.commit()

    print("Warehouse records inserted successfully.")
    print("Records inserted:", len(warehouse_records))

except Exception as e:
    conn.rollback()

    print("Warehouse record insertion failed.")
    print("Error:", e)

Records prepared for insertion: 3
Warehouse records inserted successfully.
Records inserted: 3


## 6.6 Validate Row Count

This section validates that the number of records inserted into
`commerce.warehouse` matches the number of records in the
Database-Ready Warehouse dataset.

This confirms that all prepared records were successfully loaded
into PostgreSQL.

In [28]:
# =========================================================
# 6.6 Validate Row Count
# =========================================================

# Retrieve the actual row count from PostgreSQL

cursor.execute("""
    SELECT COUNT(*)
    FROM commerce.warehouse;
""")

database_row_count = cursor.fetchone()[0]

source_row_count = len(
    zoho_inventory_warehouses_db_ready
)

row_count_matches = (
    database_row_count == source_row_count
)


# Display validation results

print("Warehouse Row Count Validation")
print("=" * 60)

print("Database-Ready rows:", source_row_count)
print("PostgreSQL rows:", database_row_count)
print("Row counts match:", row_count_matches)

Warehouse Row Count Validation
Database-Ready rows: 3
PostgreSQL rows: 3
Row counts match: True


## 6.7 Retrieve Records

This section retrieves the records from `commerce.warehouse`
after loading.

This provides a direct verification that the Warehouse records
were successfully stored and can be retrieved from PostgreSQL.

In [29]:
# =========================================================
# 6.7 Retrieve Records
# =========================================================

# Retrieve all Warehouse records from PostgreSQL

cursor.execute("""
    SELECT
        warehouse_id,
        warehouse_name,
        warehouse_type,
        address,
        city,
        country,
        warehouse_manager,
        is_active,
        created_date,
        created_by,
        updated_date,
        updated_by
    FROM commerce.warehouse
    ORDER BY warehouse_id;
""")

warehouse_database_records = cursor.fetchall()


# Display the retrieved records

print("Warehouse Records Retrieved from PostgreSQL")
print("=" * 80)

for record in warehouse_database_records:
    print(record)

print("\nTotal records retrieved:", len(warehouse_database_records))

Warehouse Records Retrieved from PostgreSQL
('WH001', 'Dubai Main Warehouse', 'General Storage', 'Al Quoz, Dubai', 'Dubai', 'UAE', 'Ahmed Saleh', True, datetime.datetime(2026, 7, 1, 0, 0), 'admin', datetime.datetime(2026, 7, 15, 0, 0), 'admin')
('WH002', 'Abu Dhabi Warehouse', 'General Storage', 'Mussafah, Abu Dhabi', 'Abu Dhabi', 'UAE', 'Fatima Noor', True, datetime.datetime(2026, 7, 2, 0, 0), 'admin', datetime.datetime(2026, 7, 15, 0, 0), 'admin')
('WH003', 'Sharjah Returns Hub', 'Returns', 'Sharjah Industrial Area', 'Sharjah', 'UAE', 'Rashid Khan', True, datetime.datetime(2026, 7, 3, 0, 0), 'admin', datetime.datetime(2026, 7, 14, 0, 0), 'admin')

Total records retrieved: 3


## 6.8 Source → Database Reconciliation

This section compares the Database-Ready Warehouse dataset with
the records retrieved from PostgreSQL.

The reconciliation verifies that:

- The same number of records exists in both sources.
- All Warehouse records were loaded successfully.
- The database values match the Database-Ready source data.

This provides record-level confirmation that the Warehouse data was
loaded correctly from the Database-Ready dataset into PostgreSQL.

In [30]:
# =========================================================
# 6.8 Source → Database Reconciliation
# =========================================================

# Convert PostgreSQL records into a DataFrame

warehouse_database_df = pd.DataFrame(
    warehouse_database_records,
    columns=zoho_inventory_warehouses_db_ready.columns
)


# Sort both datasets by Primary Key before comparison

source_reconciliation_df = (
    zoho_inventory_warehouses_db_ready
    .sort_values("warehouse_id")
    .reset_index(drop=True)
)

database_reconciliation_df = (
    warehouse_database_df
    .sort_values("warehouse_id")
    .reset_index(drop=True)
)


# Compare the source and database datasets

datasets_match = (
    source_reconciliation_df.equals(
        database_reconciliation_df
    )
)


# Display reconciliation results

print("Warehouse Source → Database Reconciliation")
print("=" * 60)

print("Database-Ready rows:", len(source_reconciliation_df))
print("PostgreSQL rows:", len(database_reconciliation_df))

print("\nDatasets match:", datasets_match)

Warehouse Source → Database Reconciliation
Database-Ready rows: 3
PostgreSQL rows: 3

Datasets match: True


## 6.9 Database Integrity Validation

This section performs final database integrity checks on
`commerce.warehouse`.

The validation confirms:

- No duplicate Primary Key values exist.
- No NULL Primary Key values exist.
- Required columns contain no NULL values.
- The total database record count is correct.

This provides final confirmation that the Warehouse table is ready
for use by downstream LJ Dev Commerce processes.

In [31]:
# =========================================================
# 6.9 Database Integrity Validation
# =========================================================

# Check for duplicate Primary Key values

cursor.execute("""
    SELECT warehouse_id, COUNT(*)
    FROM commerce.warehouse
    GROUP BY warehouse_id
    HAVING COUNT(*) > 1;
""")

duplicate_primary_keys = cursor.fetchall()


# Check for NULL Primary Key values

cursor.execute("""
    SELECT COUNT(*)
    FROM commerce.warehouse
    WHERE warehouse_id IS NULL;
""")

null_primary_key_count = cursor.fetchone()[0]


# Check for NULL values in required columns

cursor.execute("""
    SELECT COUNT(*)
    FROM commerce.warehouse
    WHERE warehouse_id IS NULL
       OR warehouse_name IS NULL
       OR warehouse_type IS NULL
       OR address IS NULL
       OR city IS NULL
       OR country IS NULL
       OR warehouse_manager IS NULL
       OR is_active IS NULL
       OR created_date IS NULL
       OR created_by IS NULL
       OR updated_date IS NULL
       OR updated_by IS NULL;
""")

required_null_count = cursor.fetchone()[0]


# Final database integrity result

database_integrity_passed = (
    len(duplicate_primary_keys) == 0
    and null_primary_key_count == 0
    and required_null_count == 0
)


# Display results

print("Warehouse Database Integrity Validation")
print("=" * 60)

print(
    "Duplicate Primary Keys:",
    len(duplicate_primary_keys)
)

print(
    "NULL Primary Keys:",
    null_primary_key_count
)

print(
    "Rows with NULL required values:",
    required_null_count
)

print("\nDatabase integrity passed:", database_integrity_passed)

Warehouse Database Integrity Validation
Duplicate Primary Keys: 0
NULL Primary Keys: 0
Rows with NULL required values: 0

Database integrity passed: True


## 6.10 Close Connection

This section closes the PostgreSQL cursor and database connection.

Closing the connection properly releases database resources and marks
the completion of the Warehouse ETL and PostgreSQL loading process.

In [32]:
# Close the PostgreSQL cursor and connection

cursor.close()
conn.close()

print("PostgreSQL cursor closed successfully.")
print("PostgreSQL connection closed successfully.")
print("\nWarehouse ETL and PostgreSQL loading process completed.")

PostgreSQL cursor closed successfully.
PostgreSQL connection closed successfully.

Warehouse ETL and PostgreSQL loading process completed.
